In [7]:
import pandas as pd
import os
import glob
from sklearn.feature_extraction.text import TfidfVectorizer
from bs4 import BeautifulSoup
import re

In [ ]:
# ฟังก์ชันทำความสะอาดข้อความ
def clean_text(text):
    text = BeautifulSoup(str(text), "html.parser").get_text()
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    stopwords = {"and", "it", "that", "a", "so", "this", "but", "the","is","to","my","was","of","on","in","its","not","for","has"
                 ,"have","hair","skin","as","product","shampoo","use","after","does","doesnt","like","at","im","not","one","other","try","good"
                 ,"because","with","your","will","too","little", "very","bad","before","be","did","really","after","me","what","all","didnt","made",
                 "felt", "no","than","got","conditioner","again","would","arrived","from","do","been","box","are","great","bottle","am","products",
                 "made","would","amazon","when","also", "sunscreen", "best", "while","sun","just", "money", "we", "put","up","even","off","came"}
    return " ".join([w for w in text.split() if w not in stopwords])
# ตัวอย่างข้อความก่อนทำความสะอาด
raw_text = "<p>This is my <b>example</b> text, and it was on the website!</p>"

# เรียกใช้ฟังก์ชัน
cleaned = clean_text(raw_text)
print("ข้อความหลังคลีน:", cleaned)

ข้อความหลังคลีน: example text website


In [10]:
# ฟังก์ชันสร้าง TF-IDF และบันทึกผลลัพธ์
def generate_tfidf_by_group(df, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    df['clean_text'] = df['text'].astype(str).apply(clean_text)

    summary = []

    for cat in df['new_main_category'].dropna().unique():
        df_cat = df[df['new_main_category'] == cat]
        for tag in df_cat['tag'].dropna().unique():
            df_tag = df_cat[df_cat['tag'] == tag]
            for sent in df_tag['sentiment'].dropna().unique():
                df_sent = df_tag[df_tag['sentiment'] == sent]
                if not df_sent.empty:
                    texts = df_sent['clean_text'].tolist()
                    vectorizer = TfidfVectorizer(max_features=20)
                    tfidf_matrix = vectorizer.fit_transform(texts)
                    feature_names = vectorizer.get_feature_names_out()
                    scores = tfidf_matrix.mean(axis=0).A1
                    tfidf_df = pd.DataFrame({'term': feature_names, 'score': scores})
                    tfidf_df = tfidf_df.sort_values(by='score', ascending=False)

                    filename = f"{cat}_{tag}_{sent}_tfidf.csv".replace(" ", "_")
                    tfidf_df.to_csv(os.path.join(output_folder, filename), index=False)

                    summary.append({
                        'category': cat,
                        'tag': tag,
                        'sentiment': sent,
                        'review_count': len(df_sent),
                        'top_terms': ", ".join(tfidf_df['term'].head(5))
                    })

    return pd.DataFrame(summary)

In [11]:
# ใช้งาน
if __name__ == "__main__":
    input_path = "data/split_by_category"
    df_all = pd.concat([pd.read_json(f) for f in glob.glob(os.path.join(input_path, "*.json"))], ignore_index=True)

    result_df = generate_tfidf_by_group(df_all, output_folder="tfidf_output")
    result_df.to_csv("data/tfidf_output/summary_IDF1.csv", index=False)
    print("TF-IDF completed and saved.")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22052\3471654468.py:3: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(str(text), "html.parser").get_text()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22052\3471654468.py:3: MarkupResemblesLocatorWarning: The input looks more like a URL than markup. You may want to use an HTTP client like requests to get the document behind the URL, and feed that document to Beautiful Soup.
  text = BeautifulSoup(str(text), "html.parser").get_text()


TF-IDF completed and saved.


In [13]:

from wordcloud import WordCloud
import matplotlib.pyplot as plt


# ฟังก์ชันล้างข้อความ
def clean_text(text):
    text = BeautifulSoup(str(text), "html.parser").get_text()
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    stopwords = {"and", "it", "that", "a", "so", "this", "but", "the","is","to","my","was","of","on","in","its"}
    return " ".join([w for w in text.split() if w not in stopwords])

# ฟังก์ชันสร้าง TF-IDF และ WordCloud
def generate_tfidf_wordclouds(df, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    df['clean_text'] = df['text'].astype(str).apply(clean_text)

    for cat in df['new_main_category'].dropna().unique():
        df_cat = df[df['new_main_category'] == cat]
        for tag in df_cat['tag'].dropna().unique():
            df_tag = df_cat[df_cat['tag'] == tag]
            for sent in df_tag['sentiment'].dropna().unique():
                df_sent = df_tag[df_tag['sentiment'] == sent]
                if not df_sent.empty:
                    texts = df_sent['clean_text'].tolist()
                    vectorizer = TfidfVectorizer(max_features=100)
                    tfidf_matrix = vectorizer.fit_transform(texts)
                    feature_names = vectorizer.get_feature_names_out()
                    scores = tfidf_matrix.mean(axis=0).A1
                    tfidf_dict = dict(zip(feature_names, scores))

                    # สร้าง WordCloud จาก TF-IDF
                    wc = WordCloud(
                        width=800, height=400,
                        background_color='white'
                    ).generate_from_frequencies(tfidf_dict)

                    filename = f"{cat}_{tag}_{sent}_tfidf_wc.png".replace(" ", "_")
                    wc.to_file(os.path.join(output_folder, filename))



In [14]:
# โหลดข้อมูล
input_path = "data/split_by_category"
df_all = pd.concat([pd.read_json(f) for f in glob.glob(os.path.join(input_path, "*.json"))], ignore_index=True)

# สร้าง WordCloud จาก TF-IDF
generate_tfidf_wordclouds(df_all, output_folder="data/tfidf_output/tfidf_wordcloud_output1")

print("WordClouds created and saved.")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22052\3802880525.py:7: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(str(text), "html.parser").get_text()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22052\3802880525.py:7: MarkupResemblesLocatorWarning: The input looks more like a URL than markup. You may want to use an HTTP client like requests to get the document behind the URL, and feed that document to Beautiful Soup.
  text = BeautifulSoup(str(text), "html.parser").get_text()


WordClouds created and saved.
